<a href="https://colab.research.google.com/github/scoutproduction/Projects/blob/main/mamba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Machine Learning (CS 584): Final Project**
---

*Mamba: Linear-Time Sequence Modeling with Selective State Spaces*

---
Michael Castady






---



In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange, repeat

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/cs584_mamba'


class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2, dt_rank='auto',
                 dt_min=1e-3, dt_max=1e-1, dt_init_floor=1e-4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv  = d_conv
        self.expand  = expand
        self.d_inner = int(expand * d_model)
        self.dt_rank = max(1, d_model // 16) if dt_rank == 'auto' else dt_rank

        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)

        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner, out_channels=self.d_inner,
            kernel_size=d_conv, groups=self.d_inner,
            padding=d_conv - 1, bias=True,
        )

        self.x_proj  = nn.Linear(self.d_inner, self.dt_rank + 2 * self.d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        # --- FIX: scale dt_proj.weight so init dominates, not random weight ---
        dt_init_std = self.dt_rank ** -0.5
        nn.init.uniform_(self.dt_proj.weight, -dt_init_std, dt_init_std)
        self.dt_proj.weight.data.mul_(0.1)           # << shrink so bias dominates at init
        # --------------------------------------------------------------------

        dt = torch.exp(
            torch.rand(self.d_inner) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        ).clamp(min=dt_init_floor)
        inv_dt = dt + torch.log(-torch.expm1(-dt))
        with torch.no_grad():
            self.dt_proj.bias.copy_(inv_dt)
        self.dt_proj.bias._no_reinit = True          # << marker for any global init to skip

        A = repeat(torch.arange(1, d_state + 1, dtype=torch.float32), 'n -> d n', d=self.d_inner)
        self.A_log = nn.Parameter(torch.log(A))
        self.A_log._no_weight_decay = True           # << hint flag (our param groups already handle it)

        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.D._no_weight_decay = True

        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x):
        # x: (B, L, d_model) -> (B, L, d_model)
        B_, L, _ = x.shape

        xz = self.in_proj(x)                   # (B, L, 2*d_inner)
        x, z = xz.chunk(2, dim=-1)             # each (B, L, d_inner)

        # causal depthwise conv
        x = rearrange(x, 'b l d -> b d l')
        x = self.conv1d(x)[:, :, :L]
        x = rearrange(x, 'b d l -> b l d')
        x = F.silu(x)

        # selection: Δ, B, C are functions of the input
        x_dbl = self.x_proj(x)                                      # (B, L, dt_rank + 2*d_state)
        dt, B_sel, C_sel = torch.split(
            x_dbl, [self.dt_rank, self.d_state, self.d_state], dim=-1
        )
        dt = F.softplus(self.dt_proj(dt))                           # (B, L, d_inner), positive

        A = -torch.exp(self.A_log.float())                          # (d_inner, d_state)

        y = self.selective_scan(x, dt, A, B_sel, C_sel, self.D)     # (B, L, d_inner)
        y = y * F.silu(z)                                           # gate
        return self.out_proj(y)

    def selective_scan(self, u, delta, A, B, C, D):
        """
        Reference (sequential) selective scan. Correct, not fast.
        Shapes:
          u, delta: (B, L, d_inner)
          A: (d_inner, d_state); D: (d_inner,)
          B, C: (B, L, d_state)
        """
        B_sz, L, d_inner = u.shape
        d_state = A.shape[1]

        # Discretization (ZOH for A, Euler for B·u — as used in the paper's reference code spec)
        dA  = torch.exp(delta.unsqueeze(-1) * A)                          # (B, L, d_inner, d_state)
        dBu = delta.unsqueeze(-1) * B.unsqueeze(2) * u.unsqueeze(-1)      # (B, L, d_inner, d_state)

        h = torch.zeros(B_sz, d_inner, d_state, device=u.device, dtype=u.dtype)
        ys = []
        for t in range(L):
            h = dA[:, t] * h + dBu[:, t]                                  # (B, d_inner, d_state)
            ys.append(torch.einsum('bdn,bn->bd', h, C[:, t]))             # (B, d_inner)
        y = torch.stack(ys, dim=1)                                        # (B, L, d_inner)
        return y + u * D

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

ssm = SelectiveSSM(d_model=128, d_state=16, d_conv=4, expand=2).to(device)
x = torch.randn(2, 64, 128, device=device, requires_grad=True)
y = ssm(x)

print("output shape:", y.shape)          # expect (2, 64, 128)
print("param count: ", sum(p.numel() for p in ssm.parameters()))
loss = y.sum()
loss.backward()
print("grad flows:  ", x.grad is not None and x.grad.abs().sum().item() > 0)

output shape: torch.Size([2, 64, 128])
param count:  116480
grad flows:   True


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
        self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

# quick test
norm = RMSNorm(128)
test_input = torch.randn(2, 10, 128) * 100   # deliberately huge numbers
test_output = norm(test_input)
print("input  std:", test_input.std().item())   # should be ~100
print("output std:", test_output.std().item())  # should be ~1

input  std: 99.43659210205078
output std: 1.0000418424606323


In [ ]:

class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around SelectiveSSM (paper Fig. 3 right)."""
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.ssm  = SelectiveSSM(d_model, d_state=d_state, d_conv=d_conv, expand=expand)
    def forward(self, x):
        return x + self.ssm(self.norm(x))


class MambaLM(nn.Module):
    """Decoder-only LM: embed -> N x MambaBlock -> norm -> tied LM head."""
    def __init__(self, vocab_size, d_model=128, n_layers=4,
                 d_state=16, d_conv=4, expand=2, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.embed  = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            MambaBlock(d_model, d_state, d_conv, expand) for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight  # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, ids):
        x = self.embed(ids)
        for layer in self.layers: x = layer(x)
        x = self.norm_f(x)
        return self.lm_head(x)  # (B, L, V)

In [ ]:
class TinyTransformer(nn.Module):
    """Decoder-only Transformer, parity in d_model and layer count."""
    def __init__(self, vocab_size, d_model=128, n_layers=4, n_heads=4,
                 d_ff=None, max_len=4096, pad_id=0):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.pad_id = pad_id
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos   = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=0.0, batch_first=True, norm_first=True, activation='gelu',
        )
        self.blocks  = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm_f  = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, ids):
        B, L = ids.shape
        pos = torch.arange(L, device=ids.device).unsqueeze(0).expand(B, L)
        x = self.embed(ids) + self.pos(pos)
        mask = torch.triu(torch.ones(L, L, device=ids.device, dtype=torch.bool), diagonal=1)
        x = self.blocks(x, mask=mask, is_causal=True)
        return self.lm_head(self.norm_f(x))

In [ ]:
torch.manual_seed(0)
V, L, B = 256, 64, 4
device = 'cuda' if torch.cuda.is_available() else 'cpu'

mamba = MambaLM(vocab_size=V, d_model=128, n_layers=4).to(device)
trans = TinyTransformer(vocab_size=V, d_model=128, n_layers=4, n_heads=4).to(device)

def n_params(m): return sum(p.numel() for p in m.parameters())
print(f"MambaLM params:         {n_params(mamba):,}")
print(f"TinyTransformer params: {n_params(trans):,}")
print(f"ratio (mamba/trans):    {n_params(mamba)/n_params(trans):.2f}")

# shape check
ids = torch.randint(0, V, (B, L), device=device)
print("MambaLM  logits:", mamba(ids).shape)   # (B, L, V)
print("TinyTrf  logits:", trans(ids).shape)

# overfit-one-batch test: both models should crush a single random batch
def overfit(model, steps=200):
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    x = torch.randint(0, V, (B, L), device=device)
    y = torch.randint(0, V, (B, L), device=device)
    losses = []
    for _ in range(steps):
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    return losses[0], losses[-1]

for name, m in [("MambaLM", mamba), ("TinyTrf", trans)]:
    l0, lN = overfit(m)
    print(f"{name}: loss {l0:.3f} -> {lN:.3f}   (lower is better; should drop a lot)")

MambaLM params:         499,328
TinyTransformer params: 1,350,400
ratio (mamba/trans):    0.37
MambaLM  logits: torch.Size([4, 64, 256])
TinyTrf  logits: torch.Size([4, 64, 256])


/tmp/ipykernel_1087/2308118733.py:14: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks  = nn.TransformerEncoder(layer, num_layers=n_layers)


MambaLM: loss 5.579 -> 0.002   (lower is better; should drop a lot)
TinyTrf: loss 5.556 -> 0.003   (lower is better; should drop a lot)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import precision_recall_fscore_support

PAD_ID, NOISE_ID, SEP_ID, BLANK_ID = 0, 1, 2, 3
DATA_IDS = list(range(4, 16))
VOCAB_SIZE = 16

print(f"vocab size:   {VOCAB_SIZE}")
print(f"special ids:  PAD={PAD_ID}, NOISE={NOISE_ID}, SEP={SEP_ID}, BLANK={BLANK_ID}")
print(f"data ids:     {DATA_IDS}  ({len(DATA_IDS)} classes)")

vocab size:   16
special ids:  PAD=0, NOISE=1, SEP=2, BLANK=3
data ids:     [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]  (12 classes)


In [ ]:
def make_selective_copy_batch(batch_size, seq_len=64, n_data=4, device='cpu'):
    total_len = seq_len + 1 + n_data
    inp  = torch.full((batch_size, total_len), NOISE_ID, dtype=torch.long)
    tgt  = torch.full((batch_size, total_len), PAD_ID,   dtype=torch.long)
    mask = torch.zeros((batch_size, total_len), dtype=torch.bool)

    for b in range(batch_size):
        positions = torch.randperm(seq_len)[:n_data].sort().values
        tokens = torch.tensor(
            [DATA_IDS[i] for i in torch.randint(0, len(DATA_IDS), (n_data,))]
        )
        inp[b, positions]      = tokens
        inp[b, seq_len]        = SEP_ID
        inp[b, seq_len + 1:]   = BLANK_ID
        tgt[b, seq_len + 1:]   = tokens
        mask[b, seq_len + 1:]  = True

    return inp.to(device), tgt.to(device), mask.to(device)


# peek at one example to sanity-check the structure
inp, tgt, mask = make_selective_copy_batch(batch_size=1, seq_len=20, n_data=3)
print("input :", inp[0].tolist())
print("target:", tgt[0].tolist())
print("mask  :", mask[0].int().tolist())
print()
print("reading it: noise/data mixed for 20 tokens, then SEP (2), then BLANKs (3)")
print("target is 0 everywhere except the answer slots, which hold the data tokens in order")

input : [14, 1, 11, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 13, 1, 1, 1, 2, 3, 3, 3]
target: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 14, 11, 13]
mask  : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1]

reading it: noise/data mixed for 20 tokens, then SEP (2), then BLANKs (3)
target is 0 everywhere except the answer slots, which hold the data tokens in order


In [ ]:
@torch.no_grad()
def evaluate(model, n_batches=20, batch_size=32, seq_len=64, n_data=4, device='cpu'):
    model.eval()
    all_pred, all_true = [], []
    for _ in range(n_batches):
        inp, tgt, mask = make_selective_copy_batch(batch_size, seq_len, n_data, device)
        logits = model(inp)
        pred = logits.argmax(-1)
        all_pred.append(pred[mask].cpu())
        all_true.append(tgt[mask].cpu())
    pred = torch.cat(all_pred).numpy()
    true = torch.cat(all_true).numpy()
    acc = (pred == true).mean()
    p, r, f1, _ = precision_recall_fscore_support(true, pred, average='macro', zero_division=0)
    return {'accuracy': acc, 'precision': p, 'recall': r, 'f1': f1}


# sanity: a random untrained model should get ~1/12 ≈ 0.083 accuracy
_rand = MambaLM(vocab_size=VOCAB_SIZE, d_model=64, n_layers=2).to(device)
m = evaluate(_rand, n_batches=5, batch_size=16, seq_len=32, n_data=3, device=device)
print(f"untrained baseline: acc={m['accuracy']:.3f}  P={m['precision']:.3f}  "
      f"R={m['recall']:.3f}  F1={m['f1']:.3f}")
print("should be near chance (~0.08); if it's already high, something is leaking")
del _rand


untrained baseline: acc=0.000  P=0.000  R=0.000  F1=0.000
should be near chance (~0.08); if it's already high, something is leaking


**Data Introduction**

In [ ]:
def train_step_lm(model, token_ids, batch_size, seq_len, opt, device, clip=1.0):
    model.train()
    inp, tgt = sample_lm_batch(token_ids, batch_size, seq_len, device)
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        logits = model(inp)
        V = logits.size(-1)
        loss = F.cross_entropy(logits.reshape(-1, V), tgt.reshape(-1))
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
    opt.step()
    return loss.item()

In [ ]:
def make_optimizer(model, lr=1e-3, weight_decay=0.1):
    """Param groups: SSM dynamics, norms, biases, and embeddings get NO weight decay."""
    no_decay_names = set()
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        # Everything below is protected from weight decay
        if (name.endswith('.bias')
            or name.endswith('A_log')
            or name.endswith('.D')
            or 'norm' in name.lower()
            or 'embed' in name.lower()):
            no_decay_names.add(name)

    decay_params, no_decay_params = [], []
    for name, p in model.named_parameters():
        (no_decay_params if name in no_decay_names else decay_params).append(p)

    return torch.optim.AdamW(
        [
            {'params': decay_params,    'weight_decay': weight_decay},
            {'params': no_decay_params, 'weight_decay': 0.0},
        ],
        lr=lr, betas=(0.9, 0.95),
    )

In [ ]:
import torch, json, os

def save_checkpoint(model, optimizer, history, name, extra=None):
    """Save model + optimizer + training history to Drive."""
    path = os.path.join(SAVE_DIR, f"{name}.pt")
    payload = {
        'model_state':     model.state_dict(),
        'optimizer_state': optimizer.state_dict() if optimizer is not None else None,
        'history':         history,
        'extra':           extra or {},
    }
    torch.save(payload, path)
    print(f"saved -> {path}  ({os.path.getsize(path)/1e6:.2f} MB)")

def load_checkpoint(model, optimizer, name, device='cpu'):
    """Load weights into model (and optimizer if provided). Returns history or None."""
    path = os.path.join(SAVE_DIR, f"{name}.pt")
    if not os.path.exists(path):
        print(f"no checkpoint at {path}")
        return None
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    if optimizer is not None and ckpt['optimizer_state'] is not None:
        optimizer.load_state_dict(ckpt['optimizer_state'])
    print(f"loaded <- {path}")
    return ckpt.get('history'), ckpt.get('extra', {})

def checkpoint_exists(name):
    return os.path.exists(os.path.join(SAVE_DIR, f"{name}.pt"))

In [ ]:
def run_selective_copy(model_cls, model_kwargs, name, steps=3000, eval_every=500,
                       batch_size=32, seq_len=64, n_data=4, lr=1e-3, device='cpu',
                       force_retrain=True):
    ckpt_name = f"selcopy_{name.lower()}"
    model = model_cls(vocab_size=VOCAB_SIZE, **model_kwargs).to(device)
    opt   = make_optimizer(model, lr=lr, weight_decay=0.1)

    if checkpoint_exists(ckpt_name) and not force_retrain:
        history, _ = load_checkpoint(model, opt, ckpt_name, device=device)
        m = evaluate(model, batch_size=batch_size, seq_len=seq_len, n_data=n_data, device=device)
        print(f"{name}: loaded | acc {m['accuracy']:.3f} | P {m['precision']:.3f} | "
              f"R {m['recall']:.3f} | F1 {m['f1']:.3f}")
        return model, history

    n_p = sum(p.numel() for p in model.parameters())
    print(f"\n=== {name} | {n_p:,} params | {steps} steps | lr={lr} ===")
    history = []
    for step in range(1, steps + 1):
        loss = train_step(model, batch_size, seq_len, n_data, opt, device)
        if step == 1 or step % eval_every == 0:
            m = evaluate(model, batch_size=batch_size, seq_len=seq_len, n_data=n_data, device=device)
            print(f"step {step:5d} | loss {loss:.4f} | acc {m['accuracy']:.3f} | "
                  f"P {m['precision']:.3f} | R {m['recall']:.3f} | F1 {m['f1']:.3f}")
            history.append({'step': step, 'loss': loss, **m})

    save_checkpoint(model, opt, history, ckpt_name,
                    extra={'model_kwargs': model_kwargs, 'task': 'selective_copy'})
    return model, history

In [ ]:
mamba_model, mamba_hist = run_selective_copy(
    MambaLM, dict(d_model=128, n_layers=4),
    name='Mamba', steps=3000, device=device, force_retrain=False,
)


=== Mamba | 468,608 params | 3000 steps | lr=0.001 ===
step     1 | loss 3.2588 | acc 0.000 | P 0.000 | R 0.000 | F1 0.000
step   500 | loss 2.4909 | acc 0.081 | P 0.007 | R 0.083 | F1 0.013
step  1000 | loss 2.1857 | acc 0.118 | P 0.072 | R 0.119 | F1 0.075
step  1500 | loss 1.6124 | acc 0.256 | P 0.161 | R 0.252 | F1 0.169
step  2000 | loss 0.3069 | acc 0.973 | P 0.973 | R 0.973 | F1 0.973
step  2500 | loss 0.0381 | acc 0.999 | P 0.999 | R 0.999 | F1 0.999
step  3000 | loss 0.0021 | acc 0.998 | P 0.998 | R 0.998 | F1 0.998
saved -> /content/drive/MyDrive/cs584_mamba/selcopy_mamba.pt  (5.68 MB)


In [ ]:
trans_model, trans_hist = run_selective_copy(
    TinyTransformer, dict(d_model=128, n_layers=4, n_heads=4, max_len=128),
    name='Transformer', steps=3000, device=device, force_retrain=True,
)


=== Transformer | 811,776 params | 3000 steps | lr=0.001 ===


/tmp/ipykernel_1087/2308118733.py:14: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks  = nn.TransformerEncoder(layer, num_layers=n_layers)


step     1 | loss 2.7766 | acc 0.082 | P 0.007 | R 0.083 | F1 0.013
step   500 | loss 0.4158 | acc 0.828 | P 0.836 | R 0.827 | F1 0.828
step  1000 | loss 0.3853 | acc 0.857 | P 0.859 | R 0.859 | F1 0.857
step  1500 | loss 0.2698 | acc 0.881 | P 0.881 | R 0.882 | F1 0.881
step  2000 | loss 0.3580 | acc 0.882 | P 0.883 | R 0.883 | F1 0.882
step  2500 | loss 0.2908 | acc 0.911 | P 0.911 | R 0.912 | F1 0.911
step  3000 | loss 0.2248 | acc 0.913 | P 0.914 | R 0.913 | F1 0.913
saved -> /content/drive/MyDrive/cs584_mamba/selcopy_transformer.pt  (9.81 MB)


In [ ]:
def final_row(hist):
    return hist[-1]

m_final = final_row(mamba_hist)
t_final = final_row(trans_hist)

print(f"{'model':<12} {'acc':>7} {'prec':>7} {'rec':>7} {'f1':>7}")
print("-" * 44)
for name, r in [('Mamba', m_final), ('Transformer', t_final)]:
    print(f"{name:<12} {r['accuracy']:>7.3f} {r['precision']:>7.3f} "
          f"{r['recall']:>7.3f} {r['f1']:>7.3f}")

model            acc    prec     rec      f1
--------------------------------------------
Mamba          0.998   0.998   0.998   0.998
Transformer    0.913   0.914   0.913   0.913


**Induction Head**

In [ ]:
IH_VOCAB_SIZE = 16
IH_KEY_ID    = 1
IH_PROBE_ID  = 0
IH_TOKENS    = list(range(2, 16))

print(f"vocab size:  {IH_VOCAB_SIZE}")
print(f"key id:      {IH_KEY_ID}")
print(f"probe id:    {IH_PROBE_ID}")
print(f"content ids: {IH_TOKENS}  ({len(IH_TOKENS)} classes, chance = {1/len(IH_TOKENS):.3f})")

vocab size:  16
key id:      1
probe id:    0
content ids: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]  (14 classes, chance = 0.071)


In [ ]:
IH_PROBE_ID = 0    # the "blank" slot where the model writes its answer
def make_induction_batch(batch_size, seq_len=64, device='cpu'):
    """
    Structure (length = seq_len + 2):
        [random tokens ...] KEY VALUE [random tokens ...] KEY PROBE
    VALUE is drawn from the SAME pool as the surrounding random tokens.
    The only way to identify VALUE is by its adjacency to the first KEY.
    Model must output VALUE at the PROBE position.
    """
    total_len = seq_len + 2
    inp  = torch.zeros((batch_size, total_len), dtype=torch.long)
    tgt  = torch.zeros((batch_size, total_len), dtype=torch.long)
    mask = torch.zeros((batch_size, total_len), dtype=torch.bool)

    n_content = len(IH_TOKENS)
    content_tensor = torch.tensor(IH_TOKENS)

    for b in range(batch_size):
        # random content fill across [0, seq_len) — same token pool as values
        rand_idx = torch.randint(0, n_content, (seq_len,))
        inp[b, :seq_len] = content_tensor[rand_idx]

        # KEY → VALUE pair at a random early position; VALUE from same pool
        first_pos = torch.randint(2, seq_len // 2, (1,)).item()
        val_id = IH_TOKENS[torch.randint(0, n_content, (1,)).item()]
        inp[b, first_pos]     = IH_KEY_ID
        inp[b, first_pos + 1] = val_id

        # second KEY and PROBE at the very end
        inp[b, seq_len]     = IH_KEY_ID
        inp[b, seq_len + 1] = IH_PROBE_ID
        tgt[b, seq_len + 1] = val_id
        mask[b, seq_len + 1] = True

    return inp.to(device), tgt.to(device), mask.to(device)


# peek
inp, tgt, mask = make_induction_batch(1, seq_len=20)
print("input :", inp[0].tolist())
print("target:", tgt[0].tolist())
print("mask  :", mask[0].int().tolist())
print()
print("look for the pattern: a 1 (KEY) early, followed by some token X.")
print("the sequence ends with 1 then 0 (KEY, PROBE). target's last slot = X.")
print("note: other occurrences of X may appear as noise — that's the point,")
print("the model must key off position relative to KEY, not token identity.")

input : [11, 13, 13, 14, 3, 1, 7, 14, 4, 6, 3, 3, 12, 11, 9, 8, 11, 11, 4, 6, 1, 0]
target: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7]
mask  : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]

look for the pattern: a 1 (KEY) early, followed by some token X.
the sequence ends with 1 then 0 (KEY, PROBE). target's last slot = X.
note: other occurrences of X may appear as noise — that's the point,
the model must key off position relative to KEY, not token identity.


In [ ]:
def train_step_ih(model, batch_size, seq_len, opt, device, clip=1.0):
    model.train()
    inp, tgt, mask = make_induction_batch(batch_size, seq_len, device)
    logits = model(inp)
    loss = F.cross_entropy(logits[mask], tgt[mask], reduction='mean')
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
    opt.step()
    return loss.item()


# smoke test
_sanity = MambaLM(vocab_size=IH_VOCAB_SIZE, d_model=64, n_layers=2).to(device)
_opt    = make_optimizer(_sanity, lr=1e-3)
_loss   = train_step_ih(_sanity, batch_size=8, seq_len=32, opt=_opt, device=device)
print(f"one step ran | loss = {_loss:.4f}  (expect ~{torch.log(torch.tensor(12.0)).item():.2f} = ln(12))")
del _sanity, _opt

one step ran | loss = 2.9980  (expect ~2.48 = ln(12))


In [ ]:
@torch.no_grad()
def evaluate_ih(model, n_batches=20, batch_size=32, seq_len=64, device='cpu'):
    model.eval()
    all_pred, all_true = [], []
    for _ in range(n_batches):
        inp, tgt, mask = make_induction_batch(batch_size, seq_len, device)
        logits = model(inp)
        pred = logits.argmax(-1)
        all_pred.append(pred[mask].cpu())
        all_true.append(tgt[mask].cpu())
    pred = torch.cat(all_pred).numpy()
    true = torch.cat(all_true).numpy()
    acc = (pred == true).mean()
    p, r, f1, _ = precision_recall_fscore_support(true, pred, average='macro', zero_division=0)
    return {'accuracy': acc, 'precision': p, 'recall': r, 'f1': f1}


# untrained baseline
_rand = MambaLM(vocab_size=IH_VOCAB_SIZE, d_model=64, n_layers=2).to(device)
m = evaluate_ih(_rand, n_batches=5, batch_size=16, seq_len=32, device=device)
print(f"untrained: acc={m['accuracy']:.3f}  P={m['precision']:.3f}  "
      f"R={m['recall']:.3f}  F1={m['f1']:.3f}")
print(f"chance is 1/12 ≈ 0.083; should be near that")
del _rand

untrained: acc=0.000  P=0.000  R=0.000  F1=0.000
chance is 1/12 ≈ 0.083; should be near that


In [ ]:
def run_induction(model_cls, model_kwargs, name, steps=3000, eval_every=500,
                  batch_size=32, seq_len=64, lr=1e-3, device='cpu',
                  force_retrain=True):
    ckpt_name = f"induction_{name.lower()}"
    model = model_cls(vocab_size=IH_VOCAB_SIZE, **model_kwargs).to(device)
    opt   = make_optimizer(model, lr=lr, weight_decay=0.1)

    if checkpoint_exists(ckpt_name) and not force_retrain:
        history, _ = load_checkpoint(model, opt, ckpt_name, device=device)
        m = evaluate_ih(model, batch_size=batch_size, seq_len=seq_len, device=device)
        print(f"{name}: loaded | acc {m['accuracy']:.3f} | P {m['precision']:.3f} | "
              f"R {m['recall']:.3f} | F1 {m['f1']:.3f}")
        return model, history

    n_p = sum(p.numel() for p in model.parameters())
    print(f"\n=== {name} | {n_p:,} params | {steps} steps | induction heads ===")
    history = []
    for step in range(1, steps + 1):
        loss = train_step_ih(model, batch_size, seq_len, opt, device)
        if step == 1 or step % eval_every == 0:
            m = evaluate_ih(model, batch_size=batch_size, seq_len=seq_len, device=device)
            print(f"step {step:5d} | loss {loss:.4f} | acc {m['accuracy']:.3f} | "
                  f"P {m['precision']:.3f} | R {m['recall']:.3f} | F1 {m['f1']:.3f}")
            history.append({'step': step, 'loss': loss, **m})

    save_checkpoint(model, opt, history, ckpt_name,
                    extra={'model_kwargs': model_kwargs, 'task': 'induction_heads'})
    return model, history

print("run_induction defined")

run_induction defined


In [ ]:
mamba_ih, mamba_ih_hist = run_induction(
    MambaLM, dict(d_model=128, n_layers=4),
    name='Mamba', steps=3000, device=device, force_retrain=True,
)


=== Mamba | 468,608 params | 3000 steps | induction heads ===
step     1 | loss 3.3107 | acc 0.000 | P 0.000 | R 0.000 | F1 0.000
step   500 | loss 2.6496 | acc 0.083 | P 0.006 | R 0.071 | F1 0.011
step  1000 | loss 2.6542 | acc 0.078 | P 0.006 | R 0.071 | F1 0.010
step  1500 | loss 2.6628 | acc 0.070 | P 0.005 | R 0.071 | F1 0.009
step  2000 | loss 2.6337 | acc 0.061 | P 0.008 | R 0.067 | F1 0.011
step  2500 | loss 2.6328 | acc 0.070 | P 0.005 | R 0.071 | F1 0.009
step  3000 | loss 2.6471 | acc 0.084 | P 0.006 | R 0.071 | F1 0.011
saved -> /content/drive/MyDrive/cs584_mamba/induction_mamba.pt  (5.68 MB)


In [ ]:
trans_ih, trans_ih_hist = run_induction(
    TinyTransformer, dict(d_model=128, n_layers=4, n_heads=4, max_len=128),
    name='Transformer', steps=3000, device=device, force_retrain=True,
)




=== Transformer | 811,776 params | 3000 steps | induction heads ===


/tmp/ipykernel_1087/2308118733.py:14: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks  = nn.TransformerEncoder(layer, num_layers=n_layers)


step     1 | loss 2.7846 | acc 0.070 | P 0.005 | R 0.071 | F1 0.009
step   500 | loss 2.6115 | acc 0.073 | P 0.031 | R 0.077 | F1 0.038
step  1000 | loss 2.2635 | acc 0.163 | P 0.111 | R 0.164 | F1 0.102
step  1500 | loss 0.5261 | acc 0.708 | P 0.586 | R 0.704 | F1 0.621
step  2000 | loss 0.6870 | acc 0.797 | P 0.740 | R 0.780 | F1 0.752
step  2500 | loss 0.2291 | acc 0.783 | P 0.692 | R 0.775 | F1 0.715
step  3000 | loss 0.2008 | acc 0.853 | P 0.776 | R 0.856 | F1 0.800
saved -> /content/drive/MyDrive/cs584_mamba/induction_transformer.pt  (9.81 MB)


In [ ]:
m_final = mamba_ih_hist[-1]
t_final = trans_ih_hist[-1]
print(f"{'model':<12} {'acc':>7} {'prec':>7} {'rec':>7} {'f1':>7}")
print("-" * 44)
for name, r in [('Mamba', m_final), ('Transformer', t_final)]:
    print(f"{name:<12} {r['accuracy']:>7.3f} {r['precision']:>7.3f} "
          f"{r['recall']:>7.3f} {r['f1']:>7.3f}")

model            acc    prec     rec      f1
--------------------------------------------
Mamba          0.084   0.006   0.071   0.011
Transformer    0.853   0.776   0.856   0.800


**Project Data-Set (Amazon Instrument Reviews)**

In [ ]:
# Colab usually has these but pin them for safety
import subprocess, sys
for pkg in ['transformers', 'datasets']:
    try:
        __import__(pkg)
        print(f"{pkg}: OK")
    except ImportError:
        print(f"installing {pkg} ...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

transformers: OK
datasets: OK


In [ ]:
from datasets import load_dataset
from transformers import GPT2TokenizerFast
import os, torch

TOKENIZER_NAME = 'gpt2'
CACHE_PATH = os.path.join(SAVE_DIR, 'wikitext2_tokens.pt')

if os.path.exists(CACHE_PATH):
    cache = torch.load(CACHE_PATH, map_location='cpu')
    train_ids = cache['train']
    valid_ids = cache['valid']
    test_ids  = cache['test']
    print(f"loaded cached tokenization from {CACHE_PATH}")
else:
    print("downloading WikiText-2 ...")
    ds = load_dataset('wikitext', 'wikitext-2-raw-v1')

    print("loading GPT-2 tokenizer ...")
    tok = GPT2TokenizerFast.from_pretrained(TOKENIZER_NAME)

    def tokenize_split(split):
        # join all non-empty lines into a single string, then tokenize
        text = '\n'.join(line for line in ds[split]['text'] if line.strip())
        ids = tok.encode(text)
        return torch.tensor(ids, dtype=torch.long)

    print("tokenizing train ...")
    train_ids = tokenize_split('train')
    print("tokenizing validation ...")
    valid_ids = tokenize_split('validation')
    print("tokenizing test ...")
    test_ids  = tokenize_split('test')

    torch.save({'train': train_ids, 'valid': valid_ids, 'test': test_ids}, CACHE_PATH)
    print(f"cached to {CACHE_PATH}")

WT_VOCAB_SIZE = 50257   # gpt-2 vocab
print(f"\ntrain tokens: {len(train_ids):,}")
print(f"valid tokens: {len(valid_ids):,}")
print(f"test  tokens: {len(test_ids):,}")
print(f"vocab size  : {WT_VOCAB_SIZE:,}")

downloading WikiText-2 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizing train ...


Token indices sequence length is longer than the specified maximum sequence length for this model (2391884 > 1024). Running this sequence through the model will result in indexing errors


tokenizing validation ...
tokenizing test ...
cached to /content/drive/MyDrive/cs584_mamba/wikitext2_tokens.pt

train tokens: 2,391,884
valid tokens: 247,289
test  tokens: 283,287
vocab size  : 50,257


In [ ]:
def sample_lm_batch(token_ids, batch_size, seq_len, device='cpu'):
    """
    Standard LM batching: pick random offsets into the flat token stream,
    return (input[t], target[t+1]) pairs of shape (batch_size, seq_len).
    """
    max_start = len(token_ids) - seq_len - 1
    starts = torch.randint(0, max_start, (batch_size,))
    inp = torch.stack([token_ids[s : s + seq_len]       for s in starts])
    tgt = torch.stack([token_ids[s + 1 : s + seq_len + 1] for s in starts])
    return inp.to(device), tgt.to(device)


# sanity
inp, tgt = sample_lm_batch(train_ids, batch_size=2, seq_len=32)
print("input  shape:", inp.shape)
print("target shape:", tgt.shape)
print("input[0][:10]:", inp[0][:10].tolist())
print("target[0][:10]:", tgt[0][:10].tolist())
print("(target should be input shifted by 1)")

input  shape: torch.Size([2, 32])
target shape: torch.Size([2, 32])
input[0][:10]: [22576, 837, 25967, 287, 11684, 16830, 1973, 7840, 2807, 837]
target[0][:10]: [837, 25967, 287, 11684, 16830, 1973, 7840, 2807, 837, 17383]
(target should be input shifted by 1)


In [ ]:
import math

def train_step_lm(model, token_ids, batch_size, seq_len, opt, device, clip=1.0):
    model.train()
    inp, tgt = sample_lm_batch(token_ids, batch_size, seq_len, device)
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        logits = model(inp)
        V = logits.size(-1)
        loss = F.cross_entropy(logits.reshape(-1, V), tgt.reshape(-1))
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
    opt.step()
    return loss.item()


@torch.no_grad()
def evaluate_lm(model, token_ids, batch_size=8, seq_len=256, n_batches=20, device='cpu'):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    for _ in range(n_batches):
        inp, tgt = sample_lm_batch(token_ids, batch_size, seq_len, device)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            logits = model(inp)
            V = logits.size(-1)
            loss = F.cross_entropy(logits.reshape(-1, V), tgt.reshape(-1), reduction='sum')
        total_loss   += loss.item()
        total_tokens += tgt.numel()
    avg_nll = total_loss / total_tokens
    ppl = math.exp(avg_nll)
    return {'loss': avg_nll, 'perplexity': ppl}


print("train_step_lm and evaluate_lm defined (bfloat16 autocast)")

train_step_lm and evaluate_lm defined (bfloat16 autocast)


In [ ]:
def run_wikitext(model_cls, model_kwargs, name,
                 steps=6000, eval_every=500, checkpoint_every=1000,
                 batch_size=16, seq_len=256, lr=3e-4,
                 device='cpu', force_retrain=True):
    ckpt_name = f"wikitext2_{name.lower()}"
    model = model_cls(vocab_size=WT_VOCAB_SIZE, **model_kwargs).to(device)
    opt   = make_optimizer(model, lr=lr, weight_decay=0.1)

    # resume logic
    history = []
    start_step = 1
    if checkpoint_exists(ckpt_name) and not force_retrain:
        loaded, extra = load_checkpoint(model, opt, ckpt_name, device=device)
        history = loaded or []
        start_step = (history[-1]['step'] + 1) if history else 1
        print(f"{name}: resumed from step {start_step - 1}")
        if start_step > steps:
            # already complete — just eval and return
            vm = evaluate_lm(model, valid_ids, batch_size=batch_size,
                             seq_len=seq_len, device=device)
            print(f"  valid loss {vm['loss']:.3f} | valid ppl {vm['perplexity']:.1f}")
            return model, history

    n_p = sum(p.numel() for p in model.parameters())
    print(f"\n=== {name} | {n_p:,} params | training from step {start_step} to {steps} ===")

    for step in range(start_step, steps + 1):
        loss = train_step_lm(model, train_ids, batch_size, seq_len, opt, device)

        if step == start_step or step % eval_every == 0 or step == steps:
            vm = evaluate_lm(model, valid_ids, batch_size=batch_size,
                             seq_len=seq_len, n_batches=20, device=device)
            print(f"step {step:5d} | train loss {loss:.3f} | "
                  f"valid loss {vm['loss']:.3f} | valid ppl {vm['perplexity']:7.1f}")
            history.append({'step': step, 'train_loss': loss, **vm})

        if step % checkpoint_every == 0:
            save_checkpoint(model, opt, history, ckpt_name,
                            extra={'model_kwargs': model_kwargs, 'task': 'wikitext2',
                                   'step': step})

    # final save + test-set perplexity
    save_checkpoint(model, opt, history, ckpt_name,
                    extra={'model_kwargs': model_kwargs, 'task': 'wikitext2', 'step': steps})

    test_m = evaluate_lm(model, test_ids, batch_size=batch_size,
                        seq_len=seq_len, n_batches=50, device=device)
    print(f"\n{name} final | test loss {test_m['loss']:.3f} | test ppl {test_m['perplexity']:.1f}")
    return model, history

print("run_wikitext defined")

run_wikitext defined


In [ ]:
mamba_wt, mamba_wt_hist = run_wikitext(
    MambaLM,
    dict(d_model=128, n_layers=4),
    name='Mamba',
    steps=2500,              # full training budget; resume-safe
    eval_every=500,          # eval costs real time; once per 500 is plenty for a curve
    checkpoint_every=1000,   # write to Drive every 1000; survives disconnects
    batch_size=32,           # doubled from 16 — better GPU utilization, smoother gradients
    seq_len=128,             # halved from 256 — sequential scan is O(L), this ~2x's throughput
    lr=3e-4,                 # unchanged; stable for LM
    device=device,
    force_retrain=True,     # resume if checkpoint exists; fresh-train if not
)


=== Mamba | 6,899,456 params | training from step 1 to 2500 ===
step     1 | train loss 10.844 | valid loss 10.810 | valid ppl 49515.4
step   500 | train loss 6.280 | valid loss 6.398 | valid ppl   600.9
step  1000 | train loss 5.874 | valid loss 6.085 | valid ppl   439.2
saved -> /content/drive/MyDrive/cs584_mamba/wikitext2_mamba.pt  (82.85 MB)
step  1500 | train loss 5.568 | valid loss 5.888 | valid ppl   360.6
step  2000 | train loss 5.402 | valid loss 5.775 | valid ppl   322.2
saved -> /content/drive/MyDrive/cs584_mamba/wikitext2_mamba.pt  (82.85 MB)
step  2500 | train loss 5.180 | valid loss 5.721 | valid ppl   305.2
saved -> /content/drive/MyDrive/cs584_mamba/wikitext2_mamba.pt  (82.85 MB)

Mamba final | test loss 5.785 | test ppl 325.4


In [ ]:
trans_wt, trans_wt_hist = run_wikitext(
    TinyTransformer,
    dict(d_model=128, n_layers=4, n_heads=4, max_len=256),
    name='Transformer',
    steps=2500,
    eval_every=500,
    checkpoint_every=1000,
    batch_size=32,
    seq_len=128,
    lr=3e-4,
    device=device,
    force_retrain=True,
)


=== Transformer | 7,259,008 params | training from step 1 to 2500 ===


/tmp/ipykernel_1087/2308118733.py:14: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks  = nn.TransformerEncoder(layer, num_layers=n_layers)


step     1 | train loss 10.833 | valid loss 10.710 | valid ppl 44781.1
step   500 | train loss 6.389 | valid loss 6.499 | valid ppl   664.6
step  1000 | train loss 5.947 | valid loss 6.172 | valid ppl   479.3
saved -> /content/drive/MyDrive/cs584_mamba/wikitext2_transformer.pt  (87.17 MB)
step  1500 | train loss 5.757 | valid loss 6.003 | valid ppl   404.5
step  2000 | train loss 5.575 | valid loss 5.864 | valid ppl   352.2
saved -> /content/drive/MyDrive/cs584_mamba/wikitext2_transformer.pt  (87.17 MB)
step  2500 | train loss 5.194 | valid loss 5.777 | valid ppl   322.7
saved -> /content/drive/MyDrive/cs584_mamba/wikitext2_transformer.pt  (87.17 MB)

Transformer final | test loss 5.780 | test ppl 323.8
